# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research Question:** Which pre-decision search signals are associated with subsequent declines in page visibilty, and how can they be used to rank pages for human review?

**Decision it supports:** This work helps human reviwers prioritize which pages to review first and decide whether content or layout updates are warranted, using observed and directional evidence rather than intuition.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:**  FlyRank warehouse release build v20260703 from Hugging Face.

**Table and grain:** I used the `fact_content_daily_performance` table, querying the February and March 2026 partitions. The working dataset contains one row per `client_hash_id` and `content_hash_id` pair after monthly aggregation.

**Feature window:** February 1–28, 2026. The ``prior_*` fields from this window were used as model features.

**Target window:** March 1–31, 2026. March impressions were used to calculate the observed `future_decline_label`.

**Excluded from features:** 
- March 2026 features were not added in training the model because they were what the model was tested on.
- `client_hash_id` and `content_hash_id` as predictive features, because they are pseudonymous identifiers. `client_hash_id` was retained only for grouped splitting and evaluation; `content_hash_id` was retained for identifying output rows. .
- `future_impressions` column: Excluded because it contains information from the target window and would cause data leakage if added to the model training data.
- `future_decline_label` column: Excluded because it is the target and as such can never be a feature

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Assumptions

- The working dataset contains one row per `client_hash_id` and `content_hash_id` pair, with February 2026 features and a March 2026 outcome.
- A row is eligible for labeling only when it has at least 100 February impressions. Rows without an observed label are excluded from model training and evaluation.
- Missing GA4 values may represent unavailable tracking rather than zero engagement.
- A missing February average position means that no measurable February search position was available; it does not mean rank zero.
- This is observational data. An observed decline does not prove that any feature caused the decline.

### Features

The model uses February-only fields that were available before the target window:

- `prior_impressions`
- `prior_clicks`
- `prior_ctr`, derived from February clicks divided by February impressions
- `prior_avg_position`
- `prior_sessions`
- `prior_engagement_rate`

Missing `prior_avg_position` values are filled with 999, while missing sessions and engagement values are filled with 0, following the modeling notebook's preprocessing. These values are used as preprocessing conventions, not as claims that the underlying measurements were truly zero.

### Label definition

`future_decline_label` is the observed March outcome. It equals 1 when a page has at least 100 February impressions and its March impressions are less than 80% of its February impressions; otherwise, eligible rows receive 0. `future_impressions` is used to construct and evaluate the label, but never as a feature.

### Baseline rule and reason codes

The transparent baseline ranks eligible client-content pairs by a rule-based estimate of future decline risk. Its score is:

```python
dataframe['baseline_score'] = (
    3 * dataframe['limited_prior_visibility']
    + dataframe['weak_position_signal']
    + dataframe['low_prior_engagement']
    + dataframe['low_click_through_rate']
)
```

The reason codes are:

- `limited_prior_visibility`: at least 100 but fewer than 1,000 prior impressions.
- `weak_position_signal`: prior average position is worse than 10.
- `low_prior_engagement`: prior engagement rate is below 30% when sessions are available.
- `low_click_through_rate`: prior clicks are low relative to prior impressions.
- `high_visibility_at_risk`: at least 1,000 prior impressions and a weak prior position; this is an interpretation of the `limited_prior_visibility` and position signals, not a separate term in the score.

A page can receive more than one reason code. The baseline is a transparent prioritization rule, not a causal model.

### Validation design

The logistic-regression model uses `GroupShuffleSplit` with a 70/30 train/test split, `client_hash_id` as the grouping variable, and `random_state=42`. Grouping keeps pages from the same client on only one side of the split, reducing client-specific memorization. Because this analysis uses one February-to-March month pair, it is not a full time-series validation across multiple target periods.

### Leakage checks

Only February `prior_*` fields are used as features. `future_impressions`, `future_decline_label`, and all other March metrics are excluded from the feature matrix because they contain target-window information. `client_hash_id` and `content_hash_id` are retained only for grouping, splitting, joining, and identifying output rows; they are not learned as predictive features. The label-derived fields `trend_direction` and `trend_pct` are also excluded.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.